# Part B: a larger 4-class brain MRI dataset (Colab or a local CPU)

Dataset: "Brain Tumor MRI Dataset" by Masoud Nickparvar on Kaggle, Version 2 (<https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset>). Four classes (glioma, meningioma, pituitary, no tumour) with a provided Training/Testing split. Licence: CC BY 4.0 (<https://creativecommons.org/licenses/by/4.0/>). The author built it from three sources: figshare, SARTAJ and Br35H. The images are not redistributed in this repo; resizing and normalisation are done by my code.

**Not a medical tool.** This is a parallel-computing and evaluation exercise. Nothing here is fit for diagnosis or any decision about a patient.

What it does, in order: count check, exact and near-duplicate check, speedup of the parallel filter pipeline on a stated subset, GLCM features + random forest, a small CNN baseline, then per-class precision/recall/F1 and confusion matrices for both models. The main result uses the provided Training/Testing split as it is.

How to run:

- **Colab:** read the licence on the dataset page, set `LICENCE_CONFIRMED = True` in the next cell and run all cells. Kaggle credentials come from Colab secrets; never write them into this notebook or any file.
- **Local CPU:** set `MRI_PART_B_LICENCE=CC-BY-4.0` and `MRI_PART_B_DATA_DIR` (the folder that holds `Training/` and `Testing/`), then run `python scripts/run_part_b.py`. Colab-only cells are skipped.

Outputs: CSV/JSON in `results/part_b/` and charts in `figures/part_b/` (no MRI images).

In [ ]:
# Licence gate: this cell stops the notebook until the licence has been checked.
import os

LICENCE_CONFIRMED = False  # set to True only after reading the dataset licence on Kaggle
# Outside Colab the confirmation comes from an environment variable instead:
# MRI_PART_B_LICENCE=CC-BY-4.0 (any other value leaves the gate closed).
if os.environ.get("MRI_PART_B_LICENCE", "").strip().upper() == "CC-BY-4.0":
    LICENCE_CONFIRMED = True
DATASET = "masoudnickparvar/brain-tumor-mri-dataset"
print(f"Dataset: https://www.kaggle.com/datasets/{DATASET} (Version 2)")
print(
    "Licence: CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Credit the author and "
    "the original sources; do not redistribute the images or commit them to any repository."
)
if not LICENCE_CONFIRMED:
    raise SystemExit(
        "STOP: licence not confirmed. Read the licence, then set LICENCE_CONFIRMED = True "
        "(or MRI_PART_B_LICENCE=CC-BY-4.0 outside Colab)."
    )

## 0. Setup

In [ ]:
# Colab or not: Colab-only cells below do nothing outside Colab.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("running on Colab" if IN_COLAB else "running outside Colab: Colab-only steps are skipped")

In [ ]:
# Install mri_pipeline. Works whether this repo is public or private.
# No credential is stored in this notebook, in the repo, or on disk.
import base64
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = "maryam843/parallel-mri-image-pipeline"
CLONE_DIR = Path("/content/parallel-mri-image-pipeline")


def _pip_install(*args, env=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    return subprocess.run(cmd, capture_output=True, text=True, env=env).returncode == 0


def _local_copy():
    """A copy of the repo uploaded to Colab or already cloned (or this checkout)."""
    for folder in (Path.cwd(), Path.cwd().parent, CLONE_DIR):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "mri_pipeline").is_dir():
            return folder
    return None


def _colab_secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def install_mri_pipeline():
    # (a) local copy
    local = _local_copy()
    if local is not None and _pip_install("-e", str(local)):
        return "(a) installed from a local copy of the repo"
    # (b) public GitHub repo; never prompt for a password
    no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if _pip_install(f"git+https://github.com/{REPO}.git", env=no_prompt):
        return "(b) installed from the public GitHub repo"
    # (c) private repo with a Colab secret named GITHUB_TOKEN. The token is passed
    # to git only through environment variables for this one command, so it never
    # appears in a URL, in .git/config, in the output or in any file.
    token = _colab_secret("GITHUB_TOKEN")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env = {
            **no_prompt,
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic}",
        }
        del token, basic
        target = CLONE_DIR if CLONE_DIR.parent.is_dir() else Path(tempfile.mkdtemp()) / "repo"
        clone = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(target)],
            capture_output=True,
            text=True,
            env=env,
        )
        del env
        if clone.returncode == 0 and _pip_install("-e", str(target)):
            return "(c) installed from a private clone using the GITHUB_TOKEN Colab secret"
    # (d) nothing worked
    raise SystemExit(
        "Could not install mri_pipeline. Choose one option and run this cell again:\n"
        "  (a) upload or clone the repo into Colab, for example to\n"
        "      /content/parallel-mri-image-pipeline;\n"
        "  (b) make the GitHub repo public, then the pip install from GitHub works;\n"
        "  (c) add a Colab secret named GITHUB_TOKEN (a read-only token for this repo) "
        "in the Colab 'Secrets' panel and allow this notebook to use it."
    )


if IN_COLAB:
    print(install_mri_pipeline())
else:
    print("Not on Colab: using the mri_pipeline installed in this environment.")

In [ ]:
# Colab only: extra tools and Kaggle credentials from Colab secrets.
if IN_COLAB:
    !pip -q install kagglehub torch torchvision
    try:
        from google.colab import userdata  # Colab secrets: KAGGLE_USERNAME, KAGGLE_KEY

        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        print("No Colab secrets set; kagglehub will use its own login flow.")

In [ ]:
# Data folder, output folders and settings.
import csv
import functools
import json
import platform
import time
from collections import Counter
from datetime import UTC, datetime
from importlib import metadata
from pathlib import Path

import numpy as np

from mri_pipeline import io, part_b, report
from mri_pipeline.experiments import machine_info, run_scaling, write_csv, write_json
from mri_pipeline.parallel import available_cores, timed_run

data_dir = part_b.data_dir_from_env()
if data_dir is not None:
    DATA_ROOT = data_dir
    data_source = f"local folder from {part_b.DATA_ENV}"
elif IN_COLAB:
    import kagglehub

    DATA_ROOT = Path(kagglehub.dataset_download(DATASET))
    data_source = "kagglehub download"
else:
    raise SystemExit(f"Set {part_b.DATA_ENV} to the folder that holds Training/ and Testing/.")

# Outputs go to results/part_b and figures/part_b of the repo (or MRI_PART_B_OUT if set).
here = Path.cwd()
repo = next((p for p in (here, *here.parents) if (p / "pyproject.toml").is_file()), here)
OUT_ROOT = Path(os.environ.get("MRI_PART_B_OUT") or repo)
RESULTS = OUT_ROOT / "results" / "part_b"
FIGURES = OUT_ROOT / "figures" / "part_b"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

QUICK = os.environ.get("MRI_PART_B_QUICK") == "1"  # tiny settings, used only by the tests
CORES = available_cores()
SEEDS = (0,) if QUICK else (0, 1, 2)
SETTINGS = {
    "scaling_images": 8 if QUICK else 200,
    "scaling_resize": 64 if QUICK else 256,
    "scaling_repeats": 1 if QUICK else 3,
    "scaling_processes": [1, 2] if QUICK else sorted({1, 2, 4, *range(6, CORES + 1, 2), CORES}),
    "scaling_subset_seed": 0,
    "hessian_mode": "fast",
    "glcm_image_size": 64 if QUICK else 256,
    "rf_trees": 20 if QUICK else 300,
    "cnn_image_size": 32 if QUICK else 128,
    "cnn_epochs": 1 if QUICK else 20,
    "cnn_batch_size": 64,
    "cnn_learning_rate": 0.001,
    "seeds": list(SEEDS),
    "quick_test_settings": QUICK,
}
started_utc = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
memory_at_start = part_b.memory_snapshot()
step_seconds = {}
print(f"data: {data_source}; logical cores: {CORES}; quick test settings: {QUICK}")
print(json.dumps(SETTINGS))

## 1. Count check

In [ ]:
t0 = time.perf_counter()
records = part_b.list_split_records(DATA_ROOT)
counts = part_b.count_check(records)
step_seconds["count_check"] = round(time.perf_counter() - t0, 2)
write_json(RESULTS / "dataset_counts.json", counts)
print(report.part_b_counts_table(RESULTS))
print("file types:", counts["extensions"])

## 2. Exact and near-duplicate check, within and across the splits

Exact duplicates: SHA-256 of the file bytes. Near duplicates: two 64-bit perceptual hashes of the grayscale image, a difference hash (dHash) and a DCT hash (pHash), compared by Hamming distance. Pair counts are reported at several thresholds for each hash. A pair counts as a near-duplicate candidate when both hashes are within 4 bits. Nothing is removed for the main result; a second evaluation leaves out the Testing images that have a candidate in Training.

In [ ]:
t0 = time.perf_counter()
hashes = {
    s: timed_run(part_b.image_hashes, [r.path for r in records[s]], worker_threads=1).results
    for s in part_b.SPLITS
}
exact = part_b.exact_duplicates(records)
near, candidates = part_b.near_duplicates(records, hashes)
step_seconds["duplicate_check"] = round(time.perf_counter() - t0, 2)
write_json(RESULTS / "duplicates.json", {"exact": exact, "near": near})
pair_fields = ["scope", "split_a", "class_a", "file_a", "split_b", "class_b", "file_b",
               "dhash_bits", "phash_bits", "exact_copy"]  # fmt: skip
with open(RESULTS / "duplicate_candidates.csv", "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=pair_fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(candidates)
shapes = Counter((h["height"], h["width"]) for s in part_b.SPLITS for h in hashes[s])
write_json(
    RESULTS / "image_sizes.json",
    {
        "distinct_shapes": len(shapes),
        "most_common_shapes": [
            {"height": hw[0], "width": hw[1], "images": n} for hw, n in shapes.most_common(5)
        ],
        "min_height": min(h for h, _ in shapes),
        "max_height": max(h for h, _ in shapes),
        "min_width": min(w for _, w in shapes),
        "max_width": max(w for _, w in shapes),
    },
)
flagged = part_b.flagged_testing_files(candidates)
print(report.part_b_duplicates_table(RESULTS))
print(f"distinct image shapes: {len(shapes)}; most common: {shapes.most_common(3)}")

## 3. Parallel filter pipeline: speedup on a stated subset

The same filter stage as Part A (six filters per image, fast mode, the seven arrays returned to the parent), on a class-balanced subset of the Training split resized to the same size, so every task costs about the same. Sequential loop vs `Pool.map` on several process counts, repeats interleaved, mean +/- sd. Timings include pool start-up.

Each pool worker gets one BLAS/OpenMP thread (`worker_threads=1`). With library defaults, every worker on Windows reserves a BLAS buffer per hardware thread, and 12 workers ran out of commit memory on this laptop. The filters themselves do not use BLAS threads, so this does not change the sequential baseline.

In [ ]:
t0 = time.perf_counter()
scaling = run_scaling(
    DATA_ROOT / "Training",
    RESULTS,
    n_images=SETTINGS["scaling_images"],
    process_counts=tuple(SETTINGS["scaling_processes"]),
    repeats=SETTINGS["scaling_repeats"],
    mode=SETTINGS["hessian_mode"],
    seed=SETTINGS["scaling_subset_seed"],
    compare_strategies=False,
    resize=SETTINGS["scaling_resize"],
    classes=part_b.class_labels(),
    worker_threads=1,
)
step_seconds["scaling"] = round(time.perf_counter() - t0, 2)
print(report.scaling_table(RESULTS, where="results/part_b"))
print("parallel outputs identical to sequential:", scaling["meta"]["parallel_equals_sequential"])

## 4. GLCM features + random forest

Each worker loads one image, resizes it, applies the six filters (fast mode) and returns the 168 GLCM features (min-max rescale), so only a path goes in and 168 numbers come back. The random forest is trained on the whole Training split and scored on the Testing split, once per seed.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

train_recs, test_recs = records["Training"], records["Testing"]
y_train = np.array([r.label for r in train_recs])
y_test = np.array([r.label for r in test_recs])
test_keys = [f"{r.class_name}/{r.path.name}" for r in test_recs]
keep = np.array([k not in flagged for k in test_keys])
EVAL_SETS = {"test_as_provided": np.ones(len(test_recs), bool), "test_without_flagged": keep}
metric_rows, matrices, fit_seconds = [], {}, {}


def evaluate(model_name, seed, pred):
    for eval_set, mask in EVAL_SETS.items():
        for row in part_b.per_class_rows(y_test[mask], pred[mask]):
            metric_rows.append({"model": model_name, "eval_set": eval_set, "seed": seed, **row})
        matrices.setdefault(model_name, {}).setdefault(eval_set, {})[str(seed)] = part_b.confusion(
            y_test[mask], pred[mask]
        )


worker = functools.partial(
    part_b.glcm_from_path, size=SETTINGS["glcm_image_size"], mode=SETTINGS["hessian_mode"]
)
extracted = timed_run(
    worker, [r.path for r in train_recs + test_recs], processes=CORES, worker_threads=1
)
X = np.stack(extracted.results)
X_train, X_test = X[: len(train_recs)], X[len(train_recs) :]
step_seconds["glcm_features"] = round(extracted.seconds, 2)
print(f"GLCM features: {X.shape[0]} images x {X.shape[1]} features, "
      f"{extracted.seconds:.1f} s on {CORES} processes")  # fmt: skip

t0 = time.perf_counter()
for seed in SEEDS:
    t1 = time.perf_counter()
    rf = RandomForestClassifier(n_estimators=SETTINGS["rf_trees"], random_state=seed, n_jobs=-1)
    pred_rf = rf.fit(X_train, y_train).predict(X_test)
    fit_seconds[f"glcm_rf_seed_{seed}"] = round(time.perf_counter() - t1, 2)
    evaluate("glcm_rf", seed, pred_rf)
    print(f"seed {seed}: test accuracy {np.mean(pred_rf == y_test):.3f}")
step_seconds["random_forest"] = round(time.perf_counter() - t0, 2)

## 5. Small CNN baseline (sized for a laptop CPU)

Four 3x3 convolution blocks (16, 32, 64, 128 channels) with batch norm, global average pooling and one linear layer (`src/mri_pipeline/cnn.py`). Grayscale input resized to the size in `SETTINGS` and scaled to 0-1, random horizontal flips, Adam, a fixed number of epochs, trained from scratch on the whole Training split once per seed. There is no validation split and no early stopping, so the Testing split is never used to choose anything. Model weights are not saved.

In [ ]:
import torch

from mri_pipeline import cnn

t0 = time.perf_counter()
size = SETTINGS["cnn_image_size"]
x_train = cnn.to_uint8_tensor(io.load_images(train_recs, size=size))
x_test = cnn.to_uint8_tensor(io.load_images(test_recs, size=size))
history_rows = []
for seed in SEEDS:
    t1 = time.perf_counter()
    model, history = cnn.train(
        x_train,
        y_train,
        n_classes=len(part_b.CLASSES),
        epochs=SETTINGS["cnn_epochs"],
        seed=seed,
        batch_size=SETTINGS["cnn_batch_size"],
        lr=SETTINGS["cnn_learning_rate"],
    )
    pred_cnn = cnn.predict(model, x_test)
    fit_seconds[f"small_cnn_seed_{seed}"] = round(time.perf_counter() - t1, 2)
    history_rows += [{"seed": seed, **h} for h in history]
    evaluate("small_cnn", seed, pred_cnn)
    print(f"seed {seed}: {len(history)} epochs, {time.perf_counter() - t1:.0f} s, "
          f"last training loss {history[-1]['train_loss']:.3f}, "
          f"test accuracy {np.mean(pred_cnn == y_test):.3f}")  # fmt: skip
    del model
step_seconds["cnn"] = round(time.perf_counter() - t0, 2)
cnn_info = {
    "parameters": cnn.count_parameters(cnn.build_model(len(part_b.CLASSES))),
    "torch_threads": torch.get_num_threads(),
}
print(cnn_info)

## 6. Per-class results, confusion matrices and figures

In [ ]:
from mri_pipeline import part_b_figures

write_csv(RESULTS / "per_class_runs.csv", metric_rows)
write_csv(RESULTS / "per_class_summary.csv", part_b.summarise_runs(metric_rows))
write_csv(RESULTS / "cnn_history.csv", history_rows)
write_json(
    RESULTS / "confusion_matrices.json",
    {"classes": list(part_b.CLASSES), "rows": "true class", "columns": "predicted class",
     "matrices": matrices},
)  # fmt: skip
versions = {}
for pkg in ("numpy", "scipy", "scikit-image", "scikit-learn", "opencv-python-headless",
            "matplotlib", "torch"):  # fmt: skip
    try:
        versions[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        versions[pkg] = None
write_json(
    RESULTS / "run_info.json",
    {
        "dataset": {
            "title": "Brain Tumor MRI Dataset",
            "author": "Masoud Nickparvar",
            "version": 2,
            "url": "https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset",
            "licence": "CC BY 4.0",
            "source": data_source,
        },
        "main_evaluation": "the provided Training/Testing split, used as it is",
        "second_evaluation": "Testing without images that have a duplicate candidate in Training",
        "test_images": {k: int(m.sum()) for k, m in EVAL_SETS.items()},
        "seeds": list(SEEDS),
        "settings": SETTINGS,
        "cnn": cnn_info,
        "step_seconds": step_seconds,
        "fit_seconds": fit_seconds,
        "machine": {**machine_info(), "packages": versions},
        "memory_at_start": memory_at_start,
        "worker_threads": 1,
        "python_implementation": platform.python_implementation(),
        "started_utc": started_utc,
        "finished_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    },
)
figures = part_b_figures.make_all(RESULTS, FIGURES)
print("figures:", [f.name for f in figures])
print(report.part_b_models_table(RESULTS))
print()
print(report.part_b_per_class_table(RESULTS))
print()
print(report.part_b_times_table(RESULTS))

## Notes

- **Not a medical tool.** These are course-exercise models on one public dataset; nothing here should be used to make decisions about a patient.
- One provided split, three seeds: the spread over seeds shows training noise only, not how the numbers would change on another split or another hospital's scans.
- Near-duplicate candidates come from hash distances; they are not checked by eye one by one.
- Timings depend on the machine; `results/part_b/run_info.json` records it.